In [0]:
%pip install optuna lightgbm xgboost catboost statsmodels scikit-learn mlflow
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =========================================================================
# 1. IMPORTAÇÕES, PARÂMETROS E CONFIGURAÇÃO DO EXPERIMENT (SANDBOX)
# =========================================================================
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

import pandas as pd
import numpy as np

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import optuna
import warnings
from datetime import date

# Parametrização

PRODUCT_LIST = (
    "Colete Salva-Vidas 2374,Bateria Náutica 5523,Cabo Náutico 5921,"
    "Hélice de Alumínio 549,GPS Plotter 2111,Cabo Náutico 5117,"
    "Hélice de Alumínio 3700,Hélice de Alumínio 571"
)
list_products = [p.strip() for p in PRODUCT_LIST.split(",")]

N_TRIALS = 1
TEST_MONTHS = 3
QUANTILE_ALPHA = 0.85
plot_product = "TODOS"

MAX_DATE = (date.today().replace(day=1) - pd.DateOffset(days=1)).date().isoformat()
# MAX_DATE = "2026-08-31"  # override manual

print(f"MAX_DATE em uso: {MAX_DATE}")

RANDOM_SEED = 42
MODEL_NAMES = ["LightGBM", "XGBoost", "CatBoost"]

# Define o registro de modelos como local/workspace (Sem Unity Catalog nesta fase)
mlflow.set_registry_uri("databricks")
mlflow.set_experiment("/Users/adriano.takata@indicium.ai/sandbox_benchmark_demand_forecasting")


# FUNÇÕES DE MÉTRICAS

def calculate_wape(y_true, y_pred):
    """Weighted Absolute Percentage Error"""
    total_actual = np.sum(np.abs(y_true))
    if total_actual == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / total_actual

def calculate_bias(y_true, y_pred):
    """Mean Forecast Bias (Subestimando < 0 < Superestimando)"""
    return np.mean(y_pred - y_true)

def calculate_pinball_loss(y_true, y_pred, quantile=0.8):
    """Pinball Loss / Quantile Loss (Avalia nível de serviço/estoque de segurança)"""
    err = y_true - y_pred
    return np.maximum(quantile * err, (quantile - 1) * err).mean()

def wape_by_product(df, y_true_col, y_pred_col, product_col="product_name"):
    """WAPE por produto (agregando todas as lojas)."""
    out = []
    for product, g in df.groupby(product_col):
        out.append({
            "product_name": product,
            "WAPE (%)": round(calculate_wape(g[y_true_col], g[y_pred_col]) * 100, 2)
        })
    return pd.DataFrame(out).sort_values("WAPE (%)").reset_index(drop=True)


# Caso a previsão seja negativa, substitua por zero
def safe_predict(model, X):
    """Previsões de demanda não podem ser negativas."""
    return np.maximum(model.predict(X), 0)

MAX_DATE em uso: 2026-08-31


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:
# =========================================================================
# LEITURA DOS DADOS AGRUPADOS POR ESTABELECIMENTO E PRODUTO
# =========================================================================
# Cada estabelecimento terá a série histórica completa de todos os produtos.
# As features temporais são calculadas por combinação estabelecimento + produto.

# Buscar vendas mensais agrupadas por estabelecimento e produto
sales_by_state_product = (
    spark.table("prod_lh_nautical.gold.fct_sales")
    .filter(F.col("is_effective_sale") == True)
    .filter(F.col("order_date") <= F.lit(MAX_DATE))
    .join(
        spark.table("prod_lh_nautical.gold.dim_products"),
        "product_variant_id",
        "inner"
    )
    .join(
        spark.table("prod_lh_nautical.gold.dim_locations"),
        "location_id",
        "inner"
    )
    .withColumn("month_start", F.date_trunc("month", F.col("order_date")))
    .withColumn("month_num", F.month(F.col("month_start")))
    .withColumn("quarter_num", F.quarter(F.col("month_start")))
    .groupBy("month_start", "location_name", "product_name")
    .agg(
        F.sum("order_item_quantity").alias("quantity_sold"),
        F.first("month_num").alias("mes"),
        F.first("quarter_num").alias("trimestre")
    )
)

# Preencher meses faltantes com zeros após a primeira venda de cada produto por estabelecimento
# Determinar a primeira e última venda de cada combinação estabelecimento-produto
first_last_sales = (
    sales_by_state_product
    .groupBy("location_name", "product_name")
    .agg(
        F.min("month_start").alias("first_sale_month"),
        F.max("month_start").alias("last_sale_month")
    )
)

all_months = (
    sales_by_state_product
    .select(F.min("month_start").alias("min_month"))
    .collect()[0]
)

# Criar um DataFrame com todos os meses
months_list = spark.sql(f"""
    SELECT explode(sequence(
        date_trunc('month', date'{all_months.min_month}'),
        date_trunc('month', date'{MAX_DATE}'),
        interval 1 month
    )) as month_start
""").withColumn("month_num", F.month(F.col("month_start")))\
   .withColumn("quarter_num", F.quarter(F.col("month_start")))

# Cross join de estados-produtos únicos com todos os meses
state_product_combos = (
    sales_by_state_product
    .select("location_name", "product_name")
    .distinct()
)

all_state_product_months = (
    state_product_combos
    .crossJoin(months_list)
    .join(first_last_sales, ["location_name", "product_name"], "inner")
    .filter(F.col("month_start") >= F.col("first_sale_month"))
    .filter(F.col("month_start") <= F.lit(MAX_DATE))
    .select("location_name", "product_name", "month_start", "month_num", "quarter_num")
)

# Fazer left join para preencher meses sem vendas com 0
sales_by_state_product = (
    all_state_product_months
    .join(
        sales_by_state_product,
        ["location_name", "product_name", "month_start"],
        "left"
    )
    .withColumn("quantity_sold", F.coalesce(F.col("quantity_sold"), F.lit(0)))
    .withColumn("mes", F.coalesce(F.col("mes"), F.col("month_num")))
    .withColumn("trimestre", F.coalesce(F.col("trimestre"), F.col("quarter_num")))
    .select("month_start", "location_name", "product_name", "quantity_sold", "mes", "trimestre")
)

# Calcular features temporais usando Window functions
# Particiona por ESTADO e PRODUTO para que cada combinação tenha suas próprias features
window_spec = Window.partitionBy("location_name", "product_name").orderBy("month_start")
window_spec_rows_3 = window_spec.rowsBetween(-2, 0)
window_spec_rows_6 = window_spec.rowsBetween(-5, 0)
window_spec_rows_12 = window_spec.rowsBetween(-11, 0)

featured_spark_df = (
    sales_by_state_product
    .withColumn("lag_1", F.lag("quantity_sold", 1).over(window_spec))
    .withColumn("lag_2", F.lag("quantity_sold", 2).over(window_spec))
    .withColumn("lag_3", F.lag("quantity_sold", 3).over(window_spec))
    .withColumn("lag_12", F.lag("quantity_sold", 12).over(window_spec))
    .withColumn("rolling_mean_3", F.avg("quantity_sold").over(window_spec_rows_3))
    .withColumn("rolling_mean_6", F.avg("quantity_sold").over(window_spec_rows_6))
    .withColumn("rolling_std_6", F.stddev("quantity_sold").over(window_spec_rows_6))
    .withColumn("rolling_max_6", F.max("quantity_sold").over(window_spec_rows_6))
    .withColumn("rolling_max_12", F.max("quantity_sold").over(window_spec_rows_12))
    .dropna()
    .orderBy("location_name", "product_name", "month_start")
)

print(f"✓ Total de registros: {featured_spark_df.count():,}")
print(f"✓ Estabelecimentos únicos: {featured_spark_df.select('location_name').distinct().count()}")
print(f"✓ Produtos únicos: {featured_spark_df.select('product_name').distinct().count()}")
print(f"✓ Combinações estado-produto: {featured_spark_df.select('location_name', 'product_name').distinct().count():,}")

✓ Total de registros: 190,924
✓ Estabelecimentos únicos: 6
✓ Produtos únicos: 496
✓ Combinações estado-produto: 2,976


In [0]:
# =========================================================================
# PREPARAÇÃO PARA OS ALGORITMOS (PANDAS IN-MEMORY)
# =========================================================================
pdf = featured_spark_df.toPandas()

pdf["product_name"] = pdf["product_name"].astype("category")
pdf["location_name"] = pdf["location_name"].astype("category")

# Split Temporal — últimos N meses reservados para TESTE FINAL (holdout).
# Este conjunto só é usado uma vez, na avaliação final dos modelos otimizados
# nunca dentro do tuning do Optuna.
cutoff_date = pdf["month_start"].max() - pd.DateOffset(months=TEST_MONTHS)

train_df = pdf[pdf["month_start"] <= cutoff_date].reset_index(drop=True)
test_df = pdf[pdf["month_start"] > cutoff_date].reset_index(drop=True)

feature_cols = [
    "location_name", "product_name", "mes", "trimestre",
    "lag_1", "lag_2", "lag_3", "lag_12",
    "rolling_mean_3", "rolling_mean_6",
    "rolling_std_6", "rolling_max_6", "rolling_max_12"]

target_col = "quantity_sold"

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

# Sem encoding: LightGBM, XGBoost e CatBoost lidam com product_name/location_name
# como categóricas nativamente (enable_categorical / cat_features).


In [0]:
# =========================================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS COM OPTUNA
# =========================================================================
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Validação temporal DENTRO do treino (TimeSeriesSplit) — o conjunto de teste
# (X_test/y_test) não é tocado até a avaliação final, evitando
# vazamento de informação do holdout para dentro do tuning de hiperparâmetros.
N_SPLITS_CV = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)

def cv_wape(clf_builder, X, y):
    """WAPE médio via TimeSeriesSplit"""
    scores = []
    for tr_idx, val_idx in tscv.split(X):
        clf = clf_builder()
        clf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = np.maximum(clf.predict(X.iloc[val_idx]), 0)
        scores.append(calculate_wape(y.iloc[val_idx], preds))
    return float(np.mean(scores))

def create_objective(model_name):
    """Cria função objetivo para Optuna baseada no modelo selecionado."""
    # Treina para o quantil QUANTILE_ALPHA (não a média condicional)
    # em vez de RMSE/L2, que "achata" a previsão perto da média
    # e ignora os picos raros de demanda.
    def objective(trial):
        if model_name == "LightGBM":
            params = {
                "objective": "quantile",
                "alpha": QUANTILE_ALPHA,
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 15, 127),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state": RANDOM_SEED,
                "verbose": -1
            }
            builder = lambda: lgb.LGBMRegressor(**params)
            return cv_wape(builder, X_train, y_train)

        elif model_name == "XGBoost":
            params = {
                "objective": "reg:quantileerror",
                "quantile_alpha": QUANTILE_ALPHA,
                "tree_method": "hist",  # exigido por reg:quantileerror
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "enable_categorical": True,
                "random_state": RANDOM_SEED
            }
            builder = lambda: xgb.XGBRegressor(**params)
            return cv_wape(builder, X_train, y_train)

        elif model_name == "CatBoost":
            params = {
                "loss_function": f"Quantile:alpha={QUANTILE_ALPHA}",
                "iterations": trial.suggest_int("iterations", 100, 1000, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 3, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "cat_features": ["product_name", "location_name"],
                "verbose": 0,
                "random_seed": RANDOM_SEED
            }
            builder = lambda: CatBoostRegressor(**params)
            return cv_wape(builder, X_train, y_train)

    return objective

# Executar otimização para cada modelo
best_params = {}

print(f"Treinando para o quantil alpha={QUANTILE_ALPHA} (LightGBM/XGBoost/CatBoost).\n")

with mlflow.start_run(run_name="optuna_hpo_parent"):

    for model_name in MODEL_NAMES:
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
            study_name=f"optuna_{model_name}"
        )
        study.optimize(create_objective(model_name), n_trials=N_TRIALS)

        best_params[model_name] = study.best_params.copy()
        best_params[model_name]["random_state"] = RANDOM_SEED

        with mlflow.start_run(run_name=f"HPO_{model_name}", nested=True):
            mlflow.log_params({f"{model_name}_{k}": v for k, v in study.best_params.items()})
            mlflow.log_param("quantile_alpha", QUANTILE_ALPHA)
            mlflow.log_metric("best_cv_wape", study.best_value)

        print(f"\n{'='*60}")
        print(f"Modelo: {model_name}")
        print(f"Melhor WAPE (CV no treino): {study.best_value:.4f}")
        print(f"Melhores Parâmetros: {study.best_params}")

Treinando para o quantil alpha=0.85 (LightGBM/XGBoost/CatBoost).


Modelo: LightGBM
Melhor WAPE (CV no treino): 0.1556
Melhores Parâmetros: {'n_estimators': 450, 'learning_rate': 0.08927180304353628, 'num_leaves': 97, 'max_depth': 10, 'min_child_samples': 12, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665}

Modelo: XGBoost
Melhor WAPE (CV no treino): 0.3234
Melhores Parâmetros: {'n_estimators': 450, 'learning_rate': 0.08927180304353628, 'max_depth': 12, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}

Modelo: CatBoost
Melhor WAPE (CV no treino): 0.0710
Melhores Parâmetros: {'iterations': 450, 'learning_rate': 0.08927180304353628, 'depth': 8, 'l2_leaf_reg': 0.24810409748678125, 'bagging_temperature': 0.15601864044243652}


In [0]:
# =========================================================================
# TREINAR MODELOS OTIMIZADOS (TREINO COMPLETO) E AVALIAR NO HOLDOUT
# =========================================================================
optimized_models = {}
optuna_results = []

for model_name in MODEL_NAMES:
    params = best_params[model_name].copy()

    if model_name == "LightGBM":
        params["objective"] = "quantile"
        params["alpha"] = QUANTILE_ALPHA
        params["verbose"] = -1
        optimized_models[model_name] = {"model": lgb.LGBMRegressor(**params)}
    elif model_name == "XGBoost":
        params["objective"] = "reg:quantileerror"
        params["quantile_alpha"] = QUANTILE_ALPHA
        params["tree_method"] = "hist"
        params["enable_categorical"] = True
        optimized_models[model_name] = {"model": xgb.XGBRegressor(**params)}
    elif model_name == "CatBoost":
        params["loss_function"] = f"Quantile:alpha={QUANTILE_ALPHA}"
        params["cat_features"] = ["product_name", "location_name"]
        params["verbose"] = 0
        optimized_models[model_name] = {"model": CatBoostRegressor(**params)}

with mlflow.start_run(run_name="optuna_eval_parent"):

    for model_name, config in optimized_models.items():
        with mlflow.start_run(run_name=f"Optimized_{model_name}", nested=True):

            clf = config["model"]

            clf.fit(X_train, y_train)
            preds = safe_predict(clf, X_test)  # único toque no holdout: avaliação final

            wape = calculate_wape(y_test, preds)
            bias = calculate_bias(y_test, preds)
            q_loss = calculate_pinball_loss(y_test, preds, quantile=QUANTILE_ALPHA)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            mae = mean_absolute_error(y_test, preds)

            # DIAGNÓSTICO DE OVERFITTING: erro no TREINO vs. erro no TESTE
            preds_train = safe_predict(clf, X_train)
            wape_train = calculate_wape(y_train, preds_train)

            gap_pp = (wape - wape_train) * 100  # gap em pontos percentuais
            gap_relativo = (wape / wape_train - 1) * 100 if wape_train > 0 else np.nan

            mlflow.log_metrics({
                "train_wape": wape_train,
                "test_wape": wape,
                "overfit_gap_pp": gap_pp,
                "test_bias": bias,
                "test_pinball_q": q_loss,
                "test_rmse": rmse,
                "test_mae": mae
            })

            signature = infer_signature(X_test, preds)
            mlflow.sklearn.log_model(clf, artifact_path="model", signature=signature, serialization_format="cloudpickle")

            optuna_results.append({
                "Modelo": model_name,
                "WAPE Treino (%)": round(wape_train * 100, 2),
                "WAPE Teste (%)": round(wape * 100, 2),
                "WAPE (%)": round(wape * 100, 2),  # mantido p/ compatibilidade com as demais células
                "Gap Treino-Teste (p.p.)": round(gap_pp, 2),
                "BIAS (Unidades)": round(bias, 4),
                f"Pinball Loss (q{QUANTILE_ALPHA})": round(q_loss, 4),
                "MAE": round(mae, 4),
                "RMSE": round(rmse, 4)
            })

            # Veredicto por modelo, impresso na hora — não precisa esperar
            # a tabela final para saber se alguma coisa está fora do esperado.
            if wape_train == 0 and wape > 0:
                veredicto = "WAPE de treino ZERO — memorização quase certa (verificar profundidade/regularização)."
            elif gap_relativo >= 100:
                veredicto = f"Erro no teste é {gap_relativo:.0f}% maior que no treino — indício forte de overfitting."
            elif gap_relativo >= 40:
                veredicto = f"Erro no teste {gap_relativo:.0f}% maior que no treino — vale atenção, mas não é grave."
            else:
                veredicto = "Gap treino→teste dentro do esperado — sem sinal forte de overfitting."

            print(f"{model_name}: WAPE treino={wape_train*100:.2f}% | WAPE teste={wape*100:.2f}% | {veredicto}")

# TABELA COMPARATIVA DOS MODELOS OTIMIZADOS (HOLDOUT)
optuna_results_df = pd.DataFrame(optuna_results).sort_values(by="WAPE (%)").reset_index(drop=True)
print(f"BIAS positivo = modelo tende a superestimar (mais seguro p/ estoque); "
      f"negativo = tende a subestimar. Com objective=quantile alpha={QUANTILE_ALPHA}, "
      f"espera-se BIAS positivo para LightGBM/XGBoost/CatBoost.")
display(optuna_results_df)

best_model_name = optuna_results_df.iloc[0]["Modelo"]
best_config = optimized_models[best_model_name]

test_df_eval = test_df.copy()
test_df_eval["pred"] = safe_predict(best_config["model"], X_test)

print(f"\nWAPE por produto — melhor modelo: {best_model_name}")
display(wape_by_product(test_df_eval, "quantity_sold", "pred"))

2026/09/18 00:50:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-29497e38-da7d.cloud.databricks.com/ml/experiments/272288110317943/models/m-cccff95b675045f89d365f41f4a2e71e?o=7474653394852199


LightGBM: WAPE treino=12.39% | WAPE teste=18.27% | Erro no teste 47% maior que no treino — vale atenção, mas não é grave.


2026/09/18 00:51:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-29497e38-da7d.cloud.databricks.com/ml/experiments/272288110317943/models/m-5cb5bc3594da4599bbf81f28dfabd84c?o=7474653394852199


XGBoost: WAPE treino=13.01% | WAPE teste=37.43% | Erro no teste é 188% maior que no treino — indício forte de overfitting.


2026/09/18 00:52:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-29497e38-da7d.cloud.databricks.com/ml/experiments/272288110317943/models/m-d7f0381d4e334e989fdb19d7e8b57263?o=7474653394852199


CatBoost: WAPE treino=8.69% | WAPE teste=10.07% | Gap treino→teste dentro do esperado — sem sinal forte de overfitting.
BIAS positivo = modelo tende a superestimar (mais seguro p/ estoque); negativo = tende a subestimar. Com objective=quantile alpha=0.85, espera-se BIAS positivo para LightGBM/XGBoost/CatBoost.


Modelo,WAPE Treino (%),WAPE Teste (%),WAPE (%),Gap Treino-Teste (p.p.),BIAS (Unidades),Pinball Loss (q0.85),MAE,RMSE
CatBoost,8.69,10.07,10.07,1.38,0.1319,0.0254,0.1431,0.7065
LightGBM,12.39,18.27,18.27,5.87,0.2041,0.0583,0.2595,0.4797
XGBoost,13.01,37.43,37.43,24.42,0.3207,0.1536,0.5317,0.9075



WAPE por produto — melhor modelo: CatBoost


product_name,WAPE (%)
Motor de Popa 4088,0.0
Vela Mestra 7242,0.0
Âncora Bruce 2416,0.0
Bússola de Bordo 4361,0.0
Tinta Antifouling 5225,0.0
Motor de Popa 8671,0.0
Motor de Popa 4710,0.0
Âncora Bruce 5388,0.0
Vela Mestra 8105,0.0
Hélice de Alumínio 7225,0.0


In [0]:
# =========================================================================
# 4.3 EMPACOTAR O CAMPEÃO NUM PYFUNC E REGISTRAR NO UNITY CATALOG
# =========================================================================
# Por que não registrar o modelo "cru": duas coisas moram fora do objeto do
# estimador e quebrariam em produção se fossem ignoradas.
#   - safe_predict(): o np.maximum(pred, 0) que impede demanda negativa é
#     aplicado manualmente depois do .predict(). Um endpoint que servisse o
#     estimador puro devolveria valores negativos.
#   - product_name como categórica: LightGBM/XGBoost (enable_categorical=True)
#     dependem do MESMO mapeamento de categorias visto no treino; o CatBoost,
#     por sua vez, espera string pura em cat_features.
# O wrapper abaixo resolve os dois casos e passa a ser a única fronteira do
# modelo: entra DataFrame com as colunas de negócio, sai previsão já tratada.

class DemandForecastModel(mlflow.pyfunc.PythonModel):
    """Wrapper de serving do forecast de demanda."""

    def __init__(self, model, feature_cols, product_categories, location_categories, model_name, quantile_alpha):
        self.model = model
        self.feature_cols = list(feature_cols)
        self.product_categories = list(product_categories)
        self.location_categories = list(location_categories)
        self.model_name = model_name
        self.quantile_alpha = quantile_alpha

    def _prepare(self, df):
        faltando = [c for c in self.feature_cols if c not in df.columns]
        if faltando:
            raise ValueError(f"Colunas ausentes na entrada: {faltando}")

        X = df[self.feature_cols].copy()
        X["product_name"] = X["product_name"].astype(str)
        X["location_name"] = X["location_name"].astype(str)

        # CatBoost espera string em cat_features; LGBM/XGB esperam a MESMA
        # categoria (mesma ordem) usada no fit — daí guardarmos as categorias.
        if self.model_name != "CatBoost":
            X["product_name"] = pd.Categorical(
                X["product_name"], categories=self.product_categories
            )
            X["location_name"] = pd.Categorical(
                X["location_name"], categories=self.location_categories
            )

        for c in self.feature_cols:
            if c not in ("product_name", "location_name"):
                X[c] = pd.to_numeric(X[c], errors="coerce")
        return X

    def predict(self, context, model_input, params=None):
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)
        X = self._prepare(model_input)
        return np.maximum(self.model.predict(X), 0)


product_categories = sorted(pdf["product_name"].astype(str).unique().tolist())
location_categories = sorted(pdf["location_name"].astype(str).unique().tolist())

pyfunc_model = DemandForecastModel(
    model=optimized_models[best_model_name]["model"],
    feature_cols=feature_cols,
    product_categories=product_categories,
    location_categories=location_categories,
    model_name=best_model_name,
    quantile_alpha=QUANTILE_ALPHA,
)

# Teste rápido: o wrapper tem que reproduzir o mesmo número do notebook.
amostra = test_df[feature_cols].head(5).copy()
amostra["product_name"] = amostra["product_name"].astype(str)

pred_wrapper = pyfunc_model.predict(None, amostra)
pred_direto = safe_predict(optimized_models[best_model_name]["model"], X_test.head(5))

print(f"Campeão: {best_model_name}")
print(f"Wrapper : {np.round(pred_wrapper, 4)}")
print(f"Direto  : {np.round(pred_direto, 4)}")
assert np.allclose(pred_wrapper, pred_direto), (
    "Wrapper divergiu do predict original — investigar o tratamento de product_name."
)
print("\n✓ Wrapper consistente com o pipeline do notebook.")

# -------------------------------------------------------------------------
# Registrar no Unity Catalog e apontar o alias @champion
# -------------------------------------------------------------------------
from mlflow.tracking import MlflowClient
import sklearn, cloudpickle, lightgbm, xgboost, catboost

# O registry do workspace está desabilitado — usamos o UC.
mlflow.set_registry_uri("databricks-uc")

CATALOG = "dev_adriano_takata_lh_nautical"
SCHEMA = "gold"
MODEL_NAME = "demand_forecast_produtos"
UC_MODEL = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

# input_example define o schema que o endpoint vai validar nas requisições.
input_example = test_df[feature_cols].head(3).copy()
input_example["product_name"] = input_example["product_name"].astype(str)
input_example["location_name"] = input_example["location_name"].astype(str)
signature = infer_signature(input_example, pyfunc_model.predict(None, input_example))

# Dependências explícitas: o pyfunc não "enxerga" lightgbm/xgboost/catboost
# sozinho, e endpoint que sobe sem elas falha na construção do container.
pip_requirements = [
    f"mlflow=={mlflow.__version__}",
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"cloudpickle=={cloudpickle.__version__}",
    f"lightgbm=={lightgbm.__version__}",
    f"xgboost=={xgboost.__version__}",
    f"catboost=={catboost.__version__}",
]

with mlflow.start_run(run_name="register_serving_model") as run:
    mlflow.log_params({
        "champion_model": best_model_name,
        "quantile_alpha": QUANTILE_ALPHA,
        "test_months": TEST_MONTHS,
        "max_date": MAX_DATE,
        "n_features": len(feature_cols),
        "n_products": len(product_categories),
    })
    mlflow.log_metric(
        "holdout_wape_pct",
        float(optuna_results_df.loc[
            optuna_results_df["Modelo"] == best_model_name, "WAPE (%)"
        ].values[0]),
    )

    model_info = mlflow.pyfunc.log_model(
        artifact_path="serving_model",
        python_model=pyfunc_model,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
        registered_model_name=UC_MODEL,
    )

version = model_info.registered_model_version
client = MlflowClient()

# Alias em vez de número fixo: trocar de versão depois não exige mexer no endpoint.
client.set_registered_model_alias(UC_MODEL, "champion", version)
client.update_model_version(
    name=UC_MODEL,
    version=version,
    description=(
        f"Forecast de demanda mensal por produto e estabelecimento. Algoritmo: {best_model_name}, "
        f"treinado no quantil alpha={QUANTILE_ALPHA} (nível de serviço). "
        f"Saída já clipada em 0."
    ),
)

print(f"✓ Registrado: {UC_MODEL} versão {version}")
print(f"✓ Alias @champion → versão {version}")
print(f"✓ URI para consumo: models:/{UC_MODEL}@champion")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-7de126c9-d7a0-4e29-90e5-deac84b5807f/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Campeão: CatBoost
Wrapper : [0.1365 0.0622 1.0648 0.0888 0.493 ]
Direto  : [0.1365 0.0622 1.0648 0.0888 0.493 ]

✓ Wrapper consistente com o pipeline do notebook.


2026/09/18 00:55:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-29497e38-da7d.cloud.databricks.com/ml/experiments/272288110317943/models/m-325c2a57511c47b3b80be2faee8e773b?o=7474653394852199
2026/09/18 00:55:20 INFO mlflow.pyfunc: Validating input example against model signature
Registered model 'dev_adriano_takata_lh_nautical.gold.demand_forecast_produtos' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '14' of model 'dev_adriano_takata_lh_nautical.gold.demand_forecast_produtos': https://dbc-29497e38-da7d.cloud.databricks.com/explore/data/models/dev_adriano_takata_lh_nautical/gold/demand_forecast_produtos/version/14?o=7474653394852199


✓ Registrado: dev_adriano_takata_lh_nautical.gold.demand_forecast_produtos versão 14
✓ Alias @champion → versão 14
✓ URI para consumo: models:/dev_adriano_takata_lh_nautical.gold.demand_forecast_produtos@champion


In [0]:
# FUNÇÃO DE PREVISÃO RECURSIVA MULTI-STEP
import math
import numpy as np
import pandas as pd

HORIZON_MONTHS = 3  # quantos meses prever à frente
LIMIAR_ZERO = 0.5  # previsão contínua abaixo disso vira 0 em vez de ceil

def _trimestre(mes: int) -> int:
    return (mes - 1) // 3 + 1

def prever_proximos_meses(pdf_historico, modelo_wrapper, feature_cols, horizon=3, min_history=12):
    """Previsão recursiva multi-step, produto a produto.

    A cada passo do horizonte:
      1. monta as features do próximo mês a partir do histórico
         (vendas reais + previsões já geradas nesta mesma chamada);
      2. chama o modelo campeão via wrapper (já aplica o clip em 0);
      3. anexa a previsão ao histórico, para servir de lag/rolling do
         próximo passo.

    Produtos com menos de `min_history` meses ficam de fora — lag_12 não
    pode ser calculado com menos de 12 meses de histórico.
    """
    resultados = []

    for (produto, loja), hist in pdf_historico.sort_values("month_start").groupby(["product_name", "location_name"]):
        serie = hist["quantity_sold"].tolist()

        if len(serie) < min_history:
            print(f"{produto} ({loja}): {len(serie)} meses de histórico (< {min_history}) — pulando.")
            continue

        ultimo_mes = hist["month_start"].max()

        for h in range(1, horizon + 1):
            mes_alvo = ultimo_mes + pd.DateOffset(months=h)

            janela_3 = serie[-3:]
            janela_6 = serie[-6:]
            janela_12 = serie[-12:]

            rolling_mean_3 = float(np.mean(janela_3))
            rolling_mean_6 = float(np.mean(janela_6))

            linha = pd.DataFrame([{
                "product_name": produto,
                "location_name": loja,
                "mes": mes_alvo.month,
                "trimestre": _trimestre(mes_alvo.month),
                "lag_1": serie[-1],
                "lag_2": serie[-2],
                "lag_3": serie[-3],
                "lag_12": serie[-12],
                "rolling_mean_3": rolling_mean_3,
                "rolling_mean_6": rolling_mean_6,
                "rolling_std_6": float(np.std(janela_6, ddof=0)),
                "rolling_max_6": float(np.max(janela_6)),
                "rolling_max_12": float(np.max(janela_12)),
            }])[feature_cols]

            pred_continua = float(modelo_wrapper.predict(None, linha)[0])

            if pred_continua < LIMIAR_ZERO:
                pred_inteira = 0
            else:
                pred_inteira = math.ceil(pred_continua)

            resultados.append({
                "product_name": produto,
                "location_name": loja,
                "month_start": mes_alvo.normalize(),
                "demand_forcast": round(pred_continua, 4),  # mais casas p/ auditar o limiar
                "unit_demand_forcast": pred_inteira,
            })

            serie.append(pred_inteira)

    return pd.DataFrame(resultados)

# RODAR A PREVISÃO PARA O HORIZONTE CONFIGURADO
forecast_rolling_df = prever_proximos_meses(
    pdf_historico=pdf,
    modelo_wrapper=pyfunc_model,
    feature_cols=feature_cols,
    horizon=HORIZON_MONTHS,
)

ultimo_mes_fechado = pdf["month_start"].max()
print(f"Último mês fechado nos dados de treino: {ultimo_mes_fechado.strftime('%Y-%m')}")
print(f"Previsão gerada para os próximos {HORIZON_MONTHS} meses "
      f"({(ultimo_mes_fechado + pd.DateOffset(months=1)).strftime('%Y-%m')} a "
      f"{(ultimo_mes_fechado + pd.DateOffset(months=HORIZON_MONTHS)).strftime('%Y-%m')}):\n")

Sonar Transducer 5255 (Armazém Siqueira): 11 meses de histórico (< 12) — pulando.
Último mês fechado nos dados de treino: 2026-08
Previsão gerada para os próximos 3 meses (2026-09 a 2026-11):



In [0]:
%sql
DROP FUNCTION IF EXISTS dev_adriano_takata_lh_nautical.gold.demand_growth_forecast_estabelecimentos;

---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-6254175573716250>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DROP FUNCTION IF EXISTS dev_adriano_takata_lh_nautical.gold.demand_growth_forecast_estabelecimentos;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 exce

In [0]:
# PERSISTIR EM DELTA (UPSERT POR product_name + month_start)
from pyspark.sql import functions as F
from delta.tables import DeltaTable

TABELA_FORECAST_ROLLING = f"{CATALOG}.{SCHEMA}.demand_forecast_rolling_3m"

forecast_spark_df = (
    spark.createDataFrame(forecast_rolling_df)
    .withColumn("model_name", F.lit(best_model_name))
    .withColumn("model_version", F.lit(str(version)))
    .withColumn("generated_at", F.current_timestamp())
)

display(forecast_spark_df)

if spark.catalog.tableExists(TABELA_FORECAST_ROLLING):
    existing_cols = [f.name for f in spark.table(TABELA_FORECAST_ROLLING).schema.fields]
    if "location_name" not in existing_cols:
        spark.sql(f"DROP TABLE IF EXISTS {TABELA_FORECAST_ROLLING}")
        forecast_spark_df.write.saveAsTable(TABELA_FORECAST_ROLLING)
        print(f"✓ Tabela recriada com location_name: {TABELA_FORECAST_ROLLING}")
    else:
        delta_tbl = DeltaTable.forName(spark, TABELA_FORECAST_ROLLING)
        (
            delta_tbl.alias("destino")
            .merge(
                forecast_spark_df.alias("novo"),
                "destino.product_name = novo.product_name AND destino.location_name = novo.location_name AND destino.month_start = novo.month_start"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✓ Upsert em {TABELA_FORECAST_ROLLING}")
else:
    forecast_spark_df.write.saveAsTable(TABELA_FORECAST_ROLLING)
    print(f"✓ Tabela criada: {TABELA_FORECAST_ROLLING}")

display(
    spark.table(TABELA_FORECAST_ROLLING)
    .orderBy("product_name", "month_start")
)

product_name,location_name,month_start,demand_forcast,unit_demand_forcast,model_name,model_version,generated_at
Bateria Náutica 1358,Armazém Camargo,2026-09-01T00:00:00.000Z,0.0107,0,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Camargo,2026-10-01T00:00:00.000Z,0.0063,0,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Camargo,2026-11-01T00:00:00.000Z,0.9998,1,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Novaes,2026-09-01T00:00:00.000Z,0.5469,1,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Novaes,2026-10-01T00:00:00.000Z,13.3463,14,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Novaes,2026-11-01T00:00:00.000Z,4.9069,5,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Siqueira,2026-09-01T00:00:00.000Z,0.0115,0,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Siqueira,2026-10-01T00:00:00.000Z,0.0184,0,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Armazém Siqueira,2026-11-01T00:00:00.000Z,0.0184,0,CatBoost,14,2026-09-18T01:20:50.606Z
Bateria Náutica 1358,Loja Leão,2026-09-01T00:00:00.000Z,0.0,0,CatBoost,14,2026-09-18T01:20:50.606Z


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6254175573716229>, line 36
     34         print(f"✓ Upsert em {TABELA_FORECAST_ROLLING}")
     35 else:
---> 36     forecast_spark_df.write.saveAsTable(TABELA_FORECAST_ROLLING)
     37     print(f"✓ Tabela criada: {TABELA_FORECAST_ROLLING}")
     39 display(
     40     spark.table(TABELA_FORECAST_ROLLING)
     41     .orderBy("product_name", "month_start")
     42 )

File <command-6254175573716229>, line 8
      3 from delta.tables import DeltaTable
      5 TABELA_FORECAST_ROLLING = f"{CATALOG}.{SCHEMA}.demand_forecast_rolling_3m"
      7 forecast_spark_df = (
----> 8     spark.createDataFrame(forecast_rolling_df)
      9     .withColumn("model_name", F.lit(best_model_name))
     10     .withColumn("model_version", F.lit(str(version)))
     11     .withColumn("generated_at", F.current_timestamp())
     12 )
     14 displ

In [0]:
# =========================================================================
# CONCATENAR HISTÓRICO COMPLETO + PREVISÃO DE 3 MESES
# =========================================================================
# Monta um dataframe único com toda a série histórica de vendas por
# produto/estabelecimento e, no final de cada série, as 3 previsões
# geradas pelo modelo campeão.

# --- Colunas padronizadas do histórico ---
hist_cols = ["product_name", "location_name", "month_start", "quantity_sold"]
hist_df = pdf[hist_cols].copy()
hist_df["is_real"] = True

# --- Colunas padronizadas da previsão ---
fcst_cols = ["product_name", "location_name", "month_start", "unit_demand_forcast"]
fcst_df = forecast_rolling_df[fcst_cols].copy()
fcst_df["quantity_sold"] = fcst_df["unit_demand_forcast"]  # valor previsto (inteiro) para alinhar com quantity_sold
fcst_df["is_real"] = False

# --- Empilhar histórico + previsão ---
colunas_finais = [
    "product_name", "location_name", "month_start",
    "is_real", "quantity_sold"
]
serie_completa = pd.concat(
    [hist_df[colunas_finais], fcst_df[colunas_finais]],
    ignore_index=True
).sort_values(["product_name", "location_name", "month_start"]).reset_index(drop=True)

print(f"Total de linhas: {len(serie_completa)}")
print(f"  • Linhas reais:    {len(hist_df)}")
print(f"  • Linhas previstas: {len(fcst_df)}")
print()
display(serie_completa)

# =========================================================================
# PERSISTIR EM DELTA (dataset_demand_full)
# =========================================================================
TABELA_SERIE_COMPLETA = f"{CATALOG}.{SCHEMA}.dataset_demand_full"

serie_completa_spark = spark.createDataFrame(serie_completa)

if spark.catalog.tableExists(TABELA_SERIE_COMPLETA):
    existing_cols = [f.name for f in spark.table(TABELA_SERIE_COMPLETA).schema.fields]
    expected_cols = ["product_name", "location_name", "month_start", "is_real", "quantity_sold"]
    if existing_cols != expected_cols:
        spark.sql(f"DROP TABLE IF EXISTS {TABELA_SERIE_COMPLETA}")
        serie_completa_spark.write.saveAsTable(TABELA_SERIE_COMPLETA)
        print(f"✓ Tabela recriada: {TABELA_SERIE_COMPLETA}")
    else:
        delta_tbl = DeltaTable.forName(spark, TABELA_SERIE_COMPLETA)
        (
            delta_tbl.alias("destino")
            .merge(
                serie_completa_spark.alias("novo"),
                "destino.product_name = novo.product_name AND destino.location_name = novo.location_name AND destino.month_start = novo.month_start"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✓ Upsert em {TABELA_SERIE_COMPLETA}")
else:
    serie_completa_spark.write.saveAsTable(TABELA_SERIE_COMPLETA)
    print(f"✓ Tabela criada: {TABELA_SERIE_COMPLETA}")

display(
    spark.table(TABELA_SERIE_COMPLETA)
    .orderBy("product_name", "location_name", "month_start")
)

Total de linhas: 199849
  • Linhas reais:    190924
  • Linhas previstas: 8925



product_name,location_name,month_start,is_real,quantity_sold
Bateria Náutica 1358,Armazém Camargo,2021-06-01T00:00:00.000Z,true,4
Bateria Náutica 1358,Armazém Camargo,2021-07-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2021-08-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2021-09-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2021-10-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2021-11-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2021-12-01T00:00:00.000Z,true,7
Bateria Náutica 1358,Armazém Camargo,2022-01-01T00:00:00.000Z,true,3
Bateria Náutica 1358,Armazém Camargo,2022-02-01T00:00:00.000Z,true,0
Bateria Náutica 1358,Armazém Camargo,2022-03-01T00:00:00.000Z,true,10


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6254175573716230>, line 63
     61         print(f"✓ Upsert em {TABELA_SERIE_COMPLETA}")
     62 else:
---> 63     serie_completa_spark.write.saveAsTable(TABELA_SERIE_COMPLETA)
     64     print(f"✓ Tabela criada: {TABELA_SERIE_COMPLETA}")
     66 display(
     67     spark.table(TABELA_SERIE_COMPLETA)
     68     .orderBy("product_name", "location_name", "month_start")
     69 )

File <command-6254175573716230>, line 40
     35 # =========================================================================
     36 # PERSISTIR EM DELTA (dataset_demand_full)
     37 # =========================================================================
     38 TABELA_SERIE_COMPLETA = f"{CATALOG}.{SCHEMA}.dataset_demand_full"
---> 40 serie_completa_spark = spark.createDataFrame(serie_completa)
     42 if spark.catalog.tableExists(TABELA_SER